In [ ]:
# Task generation

import logging
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import wntr
from scipy.stats import beta, qmc

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout, force=True)
logging.getLogger("wntr").setLevel(logging.WARNING)
logger = logging.getLogger()


def create_seed(network_name, num_iterations, base_prime=31):
    network_int = sum(ord(c) for c in network_name) % 10000
    return (base_prime * network_int) + num_iterations


def generate_tasks(
    num_iterations,
    max_pipes,
    leak_diameter_ratio_range,
    alpha=1,
    beta_param=1,
    network_name="Hanoi",
    task_file_name="tasks.pkl",
    output_dir=Path("output"),
):
    if not leak_diameter_ratio_range[0] < leak_diameter_ratio_range[1]:
        raise ValueError("Invalid leak_diameter_ratio_range: lower bound must be less than upper bound.")
    if alpha <= 0 or beta_param <= 0:
        raise ValueError("Alpha and beta parameters must be positive.")

    output_dir.mkdir(parents=True, exist_ok=True)
    seed = create_seed(network_name, num_iterations)
    logger.info("Generating %s tasks for %s (seed=%s)", num_iterations, network_name, seed)

    np_random = np.random.RandomState(seed)

    inp_file = Path("networks") / f"{network_name}.inp"
    if not inp_file.exists():
        raise FileNotFoundError(f"Network file not found: {inp_file}")
    wn = wntr.network.WaterNetworkModel(str(inp_file))

    pipe_diameters = wn.query_link_attribute("diameter", link_type=wntr.network.Pipe)
    all_pipes = wn.pipe_name_list

    iter_samples = qmc.LatinHypercube(d=1, seed=seed).random(n=num_iterations)
    samples_per_iter = []
    total_samples_needed = 0
    for i in range(num_iterations):
        N = min(int(iter_samples[i, 0] * max_pipes) + 1, len(all_pipes))
        samples_per_iter.append(N)
        total_samples_needed += N

    diameter_params = qmc.LatinHypercube(d=1, seed=seed + 1).random(n=total_samples_needed)

    tasks = []
    leak_info = []
    sample_index = 0

    for i in range(num_iterations):
        N = samples_per_iter[i]
        pipes_to_fail = np_random.choice(all_pipes, N, replace=False)

        leaks = []
        for j in range(N):
            pipe_id = pipes_to_fail[j]
            pipe_diameter = pipe_diameters[pipe_id]

            scaled_sample = beta.ppf(diameter_params[sample_index, 0], a=alpha, b=beta_param)
            leak_diameter_ratio = scaled_sample * (leak_diameter_ratio_range[1] - leak_diameter_ratio_range[0]) + leak_diameter_ratio_range[0]
            split_position_ratio = 0.5
            sample_index += 1

            leak_diameter = pipe_diameter * leak_diameter_ratio
            leak_area = np.pi * (leak_diameter / 2) ** 2

            leak_info.append(
                {
                    "Iteration": i + 1,
                    "Pipe": pipe_id,
                    "Leak_Diameter_Ratio": leak_diameter_ratio,
                    "Leak_Position_Ratio": split_position_ratio,
                    "Leak_Diameter": leak_diameter,
                    "Leak_Area": leak_area,
                }
            )
            leaks.append((pipe_id, leak_diameter_ratio, split_position_ratio))

        tasks.append((i, leaks))

    task_file_path = output_dir / task_file_name
    with open(task_file_path, "wb") as f:
        pickle.dump(tasks, f)

    logger.info("Generated %s tasks (%s leak entries)", len(tasks), len(leak_info))
    logger.info("Task list saved to %s", task_file_path)
    return tasks, pd.DataFrame(leak_info)


if __name__ == "__main__":
    network_name = "Hanoi"  # "Hanoi", "C-Town"
    num_iterations = 1000
    max_pipes = 1
    leak_diameter_ratio_range = (np.sqrt(0.01), np.sqrt(0.05))
    alpha = 1
    beta_param = 1
    task_file_name = "tasks.pkl"
    output_dir = Path("output")

    tasks, leak_info = generate_tasks(
        num_iterations=num_iterations,
        max_pipes=max_pipes,
        leak_diameter_ratio_range=leak_diameter_ratio_range,
        alpha=alpha,
        beta_param=beta_param,
        network_name=network_name,
        task_file_name=task_file_name,
        output_dir=output_dir,
    )

## Leak Simulation

In [ ]:
# Leak simulation

import logging
import pickle
import sys
import traceback
from joblib import Parallel, delayed
from pathlib import Path

import numpy as np
import pandas as pd
import wntr
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout, force=True)
logging.getLogger("wntr").setLevel(logging.WARNING)
logger = logging.getLogger()

output_dir = Path("output")
leak_dir = output_dir / "leak"
leak_pressure_dir = leak_dir / "Leak_Pressure"
leak_pressure_dir.mkdir(parents=True, exist_ok=True)

inp_file = Path("networks/Hanoi.inp") # Hanoi.inp, C-Town.inp

wn = wntr.network.WaterNetworkModel(str(inp_file))

wn.options.time.duration = 0 * 3600
wn.options.time.hydraulic_timestep = 1 * 3600
wn.options.time.report_timestep = 1 * 3600
wn.options.hydraulic.demand_model = "PDA"

for _, junction in wn.junctions():
    junction.required_pressure = 30.0
    junction.minimum_pressure = 0
    junction.pressure_exponent = 0.5

with open("wn.pickle", "wb") as f:
    pickle.dump(wn, f)


def simulate_single_iteration(task, wn_pickled):
    iteration, leaks = task
    try:
        wn_iter = pickle.loads(wn_pickled)
        for pipe_to_fail, leak_diameter_ratio, split_position_ratio in leaks:
            pipe = wn_iter.get_link(pipe_to_fail)
            leak_diameter = pipe.diameter * leak_diameter_ratio
            leak_area = np.pi * (leak_diameter / 2) ** 2

            new_node_name = f"{pipe_to_fail}_leak_node"
            wn_iter = wntr.morph.split_pipe(
                wn_iter, pipe_to_fail, f"{pipe_to_fail}_B", new_node_name, split_at_point=split_position_ratio
            )
            wn_iter.get_node(new_node_name).add_leak(wn_iter, area=leak_area, start_time=0, end_time=24 * 3600)

        sim = wntr.sim.WNTRSimulator(wn_iter)
        results = sim.run_sim()
        return {"iteration": iteration + 1, "pressure": results.node["pressure"].astype("float32")}
    except Exception as e:
        logger.error("Iteration %s failed: %s", iteration + 1, e)
        return {"iteration": iteration + 1, "error": str(e), "traceback": traceback.format_exc()}


def simulate_leakage_parallel(tasks, n_jobs=14, process_batch_size=10000):
    logger.info("Leak tasks loaded: %s", len(tasks))
    logger.info("Leak output: %s", leak_pressure_dir)
    logger.info("Leak simulation config: n_jobs=%s, batch_size=%s", n_jobs, process_batch_size)

    total_batches = (len(tasks) + process_batch_size - 1) // process_batch_size
    digits = len(str(total_batches))

    last_completed_batch = 0
    existing_batch_files = list(leak_pressure_dir.glob("Leak_Pressure_batch*.parquet"))
    if existing_batch_files:
        batch_numbers = []
        for filename in existing_batch_files:
            try:
                batch_part = filename.stem.split("batch")[-1]
                batch_numbers.append(int(batch_part))
            except Exception:
                pass
        if batch_numbers:
            last_completed_batch = max(batch_numbers)
            logger.info("Resuming from batch %s", str(last_completed_batch).zfill(digits))

    start_iteration = last_completed_batch * process_batch_size + 1
    tasks_to_run = [task for task in tasks if task[0] + 1 >= start_iteration]
    if not tasks_to_run:
        logger.info("All tasks already completed")
        return

    with open("wn.pickle", "rb") as f:
        wn_pickled = pickle.dumps(pickle.load(f))

    for i in range(0, len(tasks_to_run), process_batch_size):
        current_batch = (start_iteration - 1) // process_batch_size + 1 + (i // process_batch_size)
        batch_num_padded = str(current_batch).zfill(digits)
        batch_pressure_path = leak_pressure_dir / f"Leak_Pressure_batch{batch_num_padded}.parquet"
        if batch_pressure_path.exists():
            continue

        batch_tasks = tasks_to_run[i : min(i + process_batch_size, len(tasks_to_run))]
        logger.info("Processing leak batch %s (%s tasks)", batch_num_padded, len(batch_tasks))
        results = Parallel(n_jobs=n_jobs, max_nbytes="500M")(
            delayed(simulate_single_iteration)(task, wn_pickled)
            for task in tqdm(batch_tasks, desc=f"Leak batch {batch_num_padded}")
        )

        successful = [r for r in results if "error" not in r]
        if not successful:
            logger.warning("No successful simulations in batch %s", batch_num_padded)
            continue

        pressure_frames = []
        for r in successful:
            pressure = r["pressure"].copy()
            pressure.insert(0, "Iteration", r["iteration"])
            pressure_frames.append(pressure)

        combined_pressure = pd.concat(pressure_frames, ignore_index=True)
        combined_pressure.to_parquet(batch_pressure_path, engine="pyarrow", index=False, compression="snappy")
        del combined_pressure, pressure_frames


task_file = output_dir / "tasks.pkl"
if not task_file.exists():
    raise FileNotFoundError("Missing output/tasks.pkl. Run Cell 1 first.")
with open(task_file, "rb") as f:
    tasks = pickle.load(f)

simulate_leakage_parallel(tasks, n_jobs=14, process_batch_size=10000)

In [ ]:
# Baseline simulation

import logging
import pickle
import sys
from pathlib import Path

import pandas as pd
import wntr
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout, force=True)
logging.getLogger("wntr").setLevel(logging.WARNING)
logger = logging.getLogger()

output_dir = Path("output")
baseline_dir = output_dir / "baseline"
baseline_pressure_dir = baseline_dir / "Baseline_Pressure"
baseline_pressure_dir.mkdir(parents=True, exist_ok=True)

leak_pressure_dir = output_dir / "leak" / "Leak_Pressure"

inp_file = Path("networks/Hanoi.inp") # Hanoi.inp, C-Town.inp
wn = wntr.network.WaterNetworkModel(str(inp_file))
original_node_names = wn.node_name_list

wn.options.time.duration = 0 * 3600
wn.options.time.hydraulic_timestep = 1 * 3600
wn.options.time.report_timestep = 1 * 3600
wn.options.hydraulic.demand_model = "PDA"

for _, junction in wn.junctions():
    junction.required_pressure = 30.0
    junction.minimum_pressure = 0
    junction.pressure_exponent = 0.5

with open("wn_baseline.pickle", "wb") as f:
    pickle.dump(wn, f)


def simulate_single_baseline_iteration(task):
    i, splits = task
    try:
        with open("wn_baseline.pickle", "rb") as f:
            wn_iter = pickle.load(f)

        for pipe_to_fail, _, split_position_ratio in splits:
            new_node_name = f"{pipe_to_fail}_split_node"
            wn_iter = wntr.morph.split_pipe(
                wn_iter, pipe_to_fail, f"{pipe_to_fail}_B", new_node_name, split_at_point=split_position_ratio
            )

        sim = wntr.sim.WNTRSimulator(wn_iter)
        results = sim.run_sim()
        return {"iteration": i + 1, "pressure": results.node["pressure"].astype("float32")}
    except Exception as e:
        logger.error("Baseline iteration %s failed: %s", i + 1, e)
        return {"iteration": i + 1, "error": str(e)}


def simulate_baseline_batched(tasks):
    if not tasks:
        return

    leak_batch_files = sorted(leak_pressure_dir.glob("Leak_Pressure_batch*.parquet"))
    if not leak_batch_files:
        return

    max_batch_num = 0
    for file_path in leak_batch_files:
        try:
            batch_part = file_path.stem.split("batch")[-1]
            max_batch_num = max(max_batch_num, int(batch_part))
        except (ValueError, IndexError):
            continue
    digits = len(str(max_batch_num))

    first_task = tasks[0]
    first = simulate_single_baseline_iteration(first_task)
    if "error" in first:
        logger.error("Baseline template simulation failed: %s", first["error"])
        return

    pressure_template = first["pressure"][original_node_names].copy()

    for leak_batch_file in tqdm(leak_batch_files, desc="Baseline batches"):
        try:
            batch_part = leak_batch_file.stem.split("batch")[-1]
            batch_num = int(batch_part)
            batch_num_padded = str(batch_num).zfill(digits)
        except (ValueError, IndexError):
            continue

        baseline_pressure_batch_path = baseline_pressure_dir / f"Baseline_Pressure_batch{batch_num_padded}.parquet"
        if baseline_pressure_batch_path.exists():
            continue

        try:
            leak_df = pd.read_parquet(leak_batch_file, columns=["Iteration"], engine="pyarrow")
            leak_iterations = leak_df["Iteration"].unique()
            del leak_df
        except Exception as e:
            logger.error("Error reading leak batch file %s: %s", leak_batch_file, e)
            continue

        batch_tasks = [tasks[it - 1] for it in leak_iterations if 0 < it <= len(tasks)]
        pressure_frames = []
        for task in tqdm(batch_tasks, desc=f"Baseline batch {batch_num_padded}"):
            i, _ = task
            current_iteration = i + 1
            df = pressure_template.copy()
            df.insert(0, "Iteration", current_iteration)
            pressure_frames.append(df)

        if pressure_frames:
            combined_pressure = pd.concat(pressure_frames, ignore_index=True)
            combined_pressure["Iteration"] = combined_pressure["Iteration"].astype("int64")
            combined_pressure.to_parquet(
                baseline_pressure_batch_path, engine="pyarrow", compression="snappy", index=False
            )
            del combined_pressure, pressure_frames


task_file = output_dir / "tasks.pkl"
if not task_file.exists():
    raise FileNotFoundError("Missing output/tasks.pkl. Run Cell 1 first.")
with open(task_file, "rb") as f:
    tasks = pickle.load(f)
logger.info("Baseline tasks loaded: %s", len(tasks))
simulate_baseline_batched(tasks)